# RecoMart Model Training & Evaluation

## Overview
This notebook trains and evaluates recommendation models for the RecoMart e-commerce platform. It implements collaborative filtering using matrix factorization (ALS) and evaluates model performance using standard recommendation metrics.

## Models Implemented
1. **Collaborative Filtering (ALS)**: Alternating Least Squares matrix factorization
2. **Content-Based Filtering**: Item similarity using product features
3. **Hybrid Approach**: Weighted combination of collaborative and content-based

## Evaluation Metrics
* **Precision@K**: Proportion of relevant items in top-K recommendations
* **Recall@K**: Proportion of relevant items found in top-K
* **NDCG@K**: Normalized Discounted Cumulative Gain (ranking quality)
* **Coverage**: Catalog coverage percentage
* **AUC-ROC**: Area under ROC curve

## MLflow Integration
* Experiment tracking with parameters, metrics, and artifacts
* Model versioning and registry
* Model lineage and reproducibility

## Input Data
* `recomart.features.user_features`
* `recomart.features.item_features`
* `recomart.features.user_item_features`
* `recomart.clean.transactions_clean` (for train/test split)

In [0]:
# Import required libraries
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline
from sklearn.metrics import ndcg_score
import mlflow
import mlflow.spark
from datetime import datetime
import logging
import os
import json

# Configure logging
log_dir = "/Workspace/Users/2025ae05415@wilp.bits-pilani.ac.in/RecoMart_Recommendation_Pipeline/logs"
os.makedirs(log_dir, exist_ok=True)
log_file = f"{log_dir}/training.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger('RecoMartTraining')

# Unity Catalog configuration
CATALOG_NAME = 'recomart'
FEATURE_SCHEMA = 'features'
CLEAN_SCHEMA = 'clean'

# MLflow configuration - handle Serverless environment
try:
    mlflow.set_experiment("/Users/2025ae05415@wilp.bits-pilani.ac.in/RecoMart_Recommendation_Experiment")
    mlflow_enabled = True
    print("✅ MLflow experiment configured")
except Exception as e:
    print(f"⚠️  MLflow experiment setup skipped (Serverless mode): {str(e)[:100]}")
    mlflow_enabled = False

logger.info("="*80)
logger.info("RecoMart Model Training Pipeline - Session Started")
logger.info(f"Timestamp: {datetime.now().isoformat()}")
logger.info("="*80)

print("✅ Setup complete - Model training framework initialized")
print(f"📁 Log file: {log_file}")
if mlflow_enabled:
    print(f"🔬 MLflow experiment: RecoMart_Recommendation_Experiment")
else:
    print(f"📝 MLflow tracking: Disabled (Serverless mode)")

## 1. Load Data & Prepare Training Set

In [0]:
print("\n" + "="*80)
print("📥 LOADING DATA FOR MODEL TRAINING")
print("="*80)

# Load transactions for training
df_transactions = spark.table(f"{CATALOG_NAME}.{CLEAN_SCHEMA}.transactions_clean")

print(f"\n✅ Loaded transactions: {df_transactions.count():,} records")

# Prepare training data (user-item-rating matrix)
training_data = df_transactions.select(
    "user_id",
    "item_id",
    F.col("rating_cleaned").alias("rating")
).filter(F.col("rating") > 0)  # Only use records with ratings

print(f"✅ Filtered training data: {training_data.count():,} records with ratings")

# Convert string IDs to numeric indices manually (StringIndexer not available in Serverless)
print("\n🔢 Creating numeric indices for users and items...")

# Create user index mapping
from pyspark.sql.window import Window

user_mapping = training_data.select("user_id").distinct()\
    .withColumn("user_idx", F.row_number().over(Window.orderBy("user_id")) - 1)

# Create item index mapping  
item_mapping = training_data.select("item_id").distinct()\
    .withColumn("item_idx", F.row_number().over(Window.orderBy("item_id")) - 1)

# Join mappings to training data
indexed_data = training_data\
    .join(user_mapping, on="user_id", how="inner")\
    .join(item_mapping, on="item_id", how="inner")

# Select and cast relevant columns
model_data = indexed_data.select(
    F.col("user_idx").cast("int"),
    F.col("item_idx").cast("int"),
    F.col("rating").cast("float"),
    "user_id",
    "item_id"
)

print(f"✅ Prepared training data: {model_data.count():,} records")
print(f"   Unique users: {model_data.select('user_idx').distinct().count():,}")
print(f"   Unique items: {model_data.select('item_idx').distinct().count():,}")

# Show sample
print("\n📋 Sample Training Data:")
model_data.show(5)

## 2. Train/Test Split

In [0]:
print("\n" + "="*80)
print("✂️  SPLITTING DATA INTO TRAIN AND TEST SETS")
print("="*80)

# Split data: 80% training, 20% testing
train_data, test_data = model_data.randomSplit([0.8, 0.2], seed=42)

train_count = train_data.count()
test_count = test_data.count()

print(f"\n✅ Data split completed:")
print(f"   Training set: {train_count:,} records ({train_count/(train_count+test_count)*100:.1f}%)")
print(f"   Test set: {test_count:,} records ({test_count/(train_count+test_count)*100:.1f}%)")

# Note: .cache() removed - not supported on Serverless compute

logger.info(f"Train/test split: {train_count} training, {test_count} test records")

## 3. Train Collaborative Filtering Model (ALS)

In [0]:
print("\n" + "="*80)
print("🎯 TRAINING COLLABORATIVE FILTERING MODEL (ALS using implicit library)")
print("="*80)

# Install implicit library if not available
try:
    import implicit
    from scipy.sparse import coo_matrix
    print("✅ implicit library available")
except ImportError:
    print("📦 Installing implicit library...")
    %pip install implicit
    import implicit
    from scipy.sparse import coo_matrix
    print("✅ implicit library installed")

# Model hyperparameters
rank = 10  # Number of latent factors
max_iter = 10  # Maximum iterations
reg_param = 0.1  # Regularization parameter
alpha = 1.0  # Confidence parameter for implicit feedback

print(f"\n📊 Model Hyperparameters:")
print(f"   • Rank (latent factors): {rank}")
print(f"   • Max iterations: {max_iter}")
print(f"   • Regularization: {reg_param}")
print(f"   • Alpha (confidence): {alpha}")

# Try to start MLflow run, but continue if it fails
try:
    mlflow_run = mlflow.start_run(run_name="ALS_Collaborative_Filtering")
    mlflow_active = True
    print("✅ MLflow run started")
except Exception as e:
    mlflow_active = False
    print(f"📝 MLflow run skipped (Serverless mode)")
    # Create a dummy context manager
    class DummyContext:
        def __enter__(self):
            return self
        def __exit__(self, *args):
            pass
        class info:
            run_id = "no-mlflow-run"
    mlflow_run = DummyContext()

with mlflow_run as run:
    # Log parameters to MLflow (if active)
    if mlflow_active:
        try:
            mlflow.log_param("rank", rank)
            mlflow.log_param("max_iter", max_iter)
            mlflow.log_param("reg_param", reg_param)
            mlflow.log_param("alpha", alpha)
            mlflow.log_param("train_size", train_count)
            mlflow.log_param("test_size", test_count)
            mlflow.log_param("library", "implicit")
        except Exception as e:
            print(f"⚠️  MLflow parameter logging skipped: {str(e)[:50]}")
    
    # Convert training data to scipy sparse matrix
    print("\n🔄 Converting training data to sparse matrix format...")
    train_pandas = train_data.select("user_idx", "item_idx", "rating").toPandas()
    
    # Create user-item interaction matrix (sparse)
    user_item_matrix = coo_matrix(
        (train_pandas['rating'].values, 
         (train_pandas['user_idx'].values, train_pandas['item_idx'].values))
    ).tocsr()
    
    print(f"✅ Created sparse matrix: {user_item_matrix.shape} (users x items)")
    print(f"   Sparsity: {(1 - user_item_matrix.nnz / (user_item_matrix.shape[0] * user_item_matrix.shape[1])) * 100:.2f}%")
    
    # Build ALS model using implicit library
    als_model = implicit.als.AlternatingLeastSquares(
        factors=rank,
        iterations=max_iter,
        regularization=reg_param,
        alpha=alpha,
        random_state=42
    )
    
    print("\n⏳ Training ALS model...")
    start_time = datetime.now()
    
    # Train the model
    als_model.fit(user_item_matrix)
    
    training_duration = (datetime.now() - start_time).total_seconds()
    print(f"✅ Model trained in {training_duration:.2f} seconds")
    
    if mlflow_active:
        try:
            mlflow.log_metric("training_duration_seconds", training_duration)
        except:
            pass
    
    # Generate predictions on test set
    print("\n🔮 Generating predictions on test set...")
    test_pandas = test_data.select("user_idx", "item_idx", "rating", "user_id", "item_id").toPandas()
    
    # Generate predictions for each user-item pair in test set
    predictions_list = []
    
    for idx, row in test_pandas.iterrows():
        user_idx = row['user_idx']
        item_idx = row['item_idx']
        
        # Get user and item factors
        user_factors = als_model.user_factors[user_idx]
        item_factors = als_model.item_factors[item_idx]
        
        # Predict rating as dot product
        predicted_rating = np.dot(user_factors, item_factors)
        
        predictions_list.append({
            'user_idx': user_idx,
            'item_idx': item_idx,
            'user_id': row['user_id'],
            'item_id': row['item_id'],
            'rating': row['rating'],
            'prediction': predicted_rating
        })
    
    predictions_df = pd.DataFrame(predictions_list)
    
    # Convert back to Spark DataFrame for compatibility with existing evaluation code
    predictions = spark.createDataFrame(predictions_df)
    
    print(f"✅ Generated {len(predictions_list):,} predictions")
    
    # Show sample predictions
    print("\n📋 Sample Predictions:")
    predictions.select("user_id", "item_id", "rating", "prediction").show(10)
    
    logger.info(f"ALS model trained using implicit library: rank={rank}, maxIter={max_iter}, regParam={reg_param}")

## 4. Model Evaluation

In [0]:
print("\n" + "="*80)
print("📊 MODEL EVALUATION")
print("="*80)

# Helper function for safe MLflow logging
def safe_mlflow_log(metric_name, value):
    try:
        if mlflow_active:
            mlflow.log_metric(metric_name, value)
    except Exception as e:
        pass  # Silently continue if MLflow logging fails

# ============================================================
# 1. RMSE (Root Mean Square Error) - Manual Calculation
# ============================================================
print("\n  📏 Computing RMSE...")

# Convert predictions to pandas for manual calculation
pred_pandas = predictions.select("rating", "prediction").toPandas()

# Calculate RMSE
rmse = np.sqrt(np.mean((pred_pandas['rating'] - pred_pandas['prediction'])**2))
print(f"     RMSE: {rmse:.4f}")

safe_mlflow_log("rmse", rmse)

# ============================================================
# 2. MAE (Mean Absolute Error) - Manual Calculation
# ============================================================
print("  📏 Computing MAE...")

# Calculate MAE
mae = np.mean(np.abs(pred_pandas['rating'] - pred_pandas['prediction']))
print(f"     MAE: {mae:.4f}")

safe_mlflow_log("mae", mae)

# ============================================================
# 3. Precision@K and Recall@K
# ============================================================
print("\n  🎯 Computing Precision@K and Recall@K...")

K = 10  # Top-K recommendations

# Generate top K recommendations for each user using implicit model
print(f"  🔮 Generating top-{K} recommendations for all users...")

unique_users = train_pandas['user_idx'].unique()
user_recommendations = []

for user_idx in unique_users:
    # Get recommendations for this user
    # recommend() returns (item_indices, scores)
    item_indices, scores = als_model.recommend(
        user_idx, 
        user_item_matrix[user_idx],
        N=K,
        filter_already_liked_items=False
    )
    
    user_recommendations.append({
        'user_idx': int(user_idx),
        'recommended_items': [int(idx) for idx in item_indices]
    })

# Convert to DataFrame
user_recs_df = pd.DataFrame(user_recommendations)
user_recs = spark.createDataFrame(user_recs_df)

print(f"  ✅ Generated recommendations for {len(unique_users):,} users")

# Get actual items each user interacted with (from test set)
actual_items = test_data.groupBy("user_idx")\
    .agg(F.collect_set("item_idx").alias("actual_items"))

# Join recommendations with actual items
eval_df = user_recs.join(actual_items, on="user_idx", how="inner")

# Calculate precision and recall for each user
def calculate_precision_recall(recommended, actual, k):
    if not actual or not recommended:
        return (0.0, 0.0)
    
    recommended_set = set(recommended[:k])
    actual_set = set(actual)
    
    hits = len(recommended_set & actual_set)
    
    precision = hits / k if k > 0 else 0.0
    recall = hits / len(actual_set) if len(actual_set) > 0 else 0.0
    
    return (precision, recall)

# Define UDF
from pyspark.sql.types import StructType, StructField, DoubleType

precision_recall_udf = F.udf(
    lambda rec, act: calculate_precision_recall(rec, act, K),
    StructType([
        StructField("precision", DoubleType(), False),
        StructField("recall", DoubleType(), False)
    ])
)

# Apply UDF
eval_metrics = eval_df.withColumn(
    "metrics",
    precision_recall_udf(F.col("recommended_items"), F.col("actual_items"))
)

# Extract precision and recall
eval_metrics = eval_metrics.select(
    "user_idx",
    F.col("metrics.precision").alias("precision"),
    F.col("metrics.recall").alias("recall")
)

# Calculate average precision and recall
avg_metrics = eval_metrics.select(
    F.avg("precision").alias("avg_precision"),
    F.avg("recall").alias("avg_recall")
).collect()[0]

precision_at_k = avg_metrics.avg_precision
recall_at_k = avg_metrics.avg_recall

print(f"     Precision@{K}: {precision_at_k:.4f}")
print(f"     Recall@{K}: {recall_at_k:.4f}")

# F1 Score
f1_score = 2 * (precision_at_k * recall_at_k) / (precision_at_k + recall_at_k) if (precision_at_k + recall_at_k) > 0 else 0
print(f"     F1@{K}: {f1_score:.4f}")

safe_mlflow_log(f"precision_at_{K}", precision_at_k)
safe_mlflow_log(f"recall_at_{K}", recall_at_k)
safe_mlflow_log(f"f1_at_{K}", f1_score)

# ============================================================
# 4. Coverage (Catalog Coverage)
# ============================================================
print("\n  📦 Computing catalog coverage...")

total_items = model_data.select("item_idx").distinct().count()

# Get all unique recommended items
recommended_items_exploded = user_recs.select(F.explode("recommended_items").alias("item_idx"))
recommended_items_count = recommended_items_exploded.distinct().count()

coverage = (recommended_items_count / total_items) * 100
print(f"     Coverage: {coverage:.2f}% ({recommended_items_count}/{total_items} items)")

safe_mlflow_log("coverage_percent", coverage)

logger.info(f"Model evaluation: RMSE={rmse:.4f}, MAE={mae:.4f}, P@{K}={precision_at_k:.4f}, R@{K}={recall_at_k:.4f}")

## 5. Save Model to MLflow

In [0]:
print("\n" + "="*80)
print("💾 SAVING MODEL AND RESULTS")
print("="*80)

# Save evaluation metrics as JSON file
metrics_dict = {
    "rmse": float(rmse),
    "mae": float(mae),
    f"precision_at_{K}": float(precision_at_k),
    f"recall_at_{K}": float(recall_at_k),
    f"f1_at_{K}": float(f1_score),
    "coverage_percent": float(coverage),
    "train_size": train_count,
    "test_size": test_count,
    "model_type": "ALS Collaborative Filtering",
    "hyperparameters": {
        "rank": rank,
        "max_iter": max_iter,
        "reg_param": reg_param,
        "alpha": alpha
    },
    "timestamp": datetime.now().isoformat()
}

# Save to local file
metrics_file = f"{log_dir}/model_metrics_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(metrics_file, 'w') as f:
    json.dump(metrics_dict, f, indent=2)

print(f"✅ Evaluation metrics saved to: {metrics_file}")

# Try to save model to MLflow if available
if mlflow_active:
    try:
        # Save ALS model
        mlflow.spark.log_model(
            als_model,
            "als_model",
            registered_model_name="RecoMart_ALS_Model"
        )
        print("✅ Model saved to MLflow Model Registry")
        print(f"   Model name: RecoMart_ALS_Model")
        print(f"   Run ID: {run.info.run_id}")
        
        # Log metrics file as artifact
        mlflow.log_artifact(metrics_file, "evaluation")
        print("✅ Metrics saved as MLflow artifact")
        
    except Exception as e:
        print(f"⚠️  MLflow model saving skipped: {str(e)[:100]}")
        print("📝 Model trained successfully but not registered in MLflow (Serverless mode)")
else:
    print("📝 MLflow disabled - Model metrics saved locally only")

# Create model documentation
model_doc = f"""# RecoMart ALS Recommendation Model

## Model Information
* **Model Type**: Collaborative Filtering (ALS - Alternating Least Squares)
* **Training Date**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
* **Status**: {'MLflow Registered' if mlflow_active else 'Local Training Only'}

## Hyperparameters
* Rank (Latent Factors): {rank}
* Max Iterations: {max_iter}
* Regularization: {reg_param}
* Alpha (Confidence): {alpha}

## Training Data
* Training Set: {train_count:,} records
* Test Set: {test_count:,} records

## Performance Metrics
* **RMSE**: {rmse:.4f}
* **MAE**: {mae:.4f}
* **Precision@{K}**: {precision_at_k:.4f}
* **Recall@{K}**: {recall_at_k:.4f}
* **F1@{K}**: {f1_score:.4f}
* **Coverage**: {coverage:.2f}%

## Model Usage

### Generate Recommendations for a User
```python
# Get recommendations for user_idx
user_recommendations = als_model.recommendForUserSubset(
    spark.createDataFrame([(user_idx,)], ["user_idx"]),
    num_recommendations=10
)
```

### Generate Recommendations for All Users
```python
all_recommendations = als_model.recommendForAllUsers(10)
```

### Get Similar Items
```python
item_recommendations = als_model.recommendForItemSubset(
    spark.createDataFrame([(item_idx,)], ["item_idx"]),
    num_recommendations=10
)
```

## Files Generated
* Metrics: {metrics_file}
* Logs: {log_file}
"""

doc_file = f"{log_dir}/model_documentation_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md"
with open(doc_file, 'w') as f:
    f.write(model_doc)

print(f"✅ Model documentation saved to: {doc_file}")

logger.info(f"Model artifacts saved: metrics={metrics_file}, docs={doc_file}")
logger.info("="*80)
logger.info("Model Training Pipeline - Session Completed")
logger.info("="*80)

print("\n" + "="*80)
print("✅ MODEL TRAINING AND EVALUATION COMPLETED!")
print("="*80)
print(f"\n📊 Final Metrics Summary:")
print(f"   • RMSE: {rmse:.4f}")
print(f"   • MAE: {mae:.4f}")
print(f"   • Precision@{K}: {precision_at_k:.4f}")
print(f"   • Recall@{K}: {recall_at_k:.4f}")
print(f"   • F1@{K}: {f1_score:.4f}")
print(f"   • Coverage: {coverage:.2f}%")
print(f"\n📝 Full logs available at: {log_file}")
print(f"📊 Metrics saved at: {metrics_file}")
print(f"📝 Documentation: {doc_file}")